> ## ⚠️ ARCHIVED — the results in this notebook are INVALID
>
> Kept as a record of how the project started. **Do not use its numbers.**
> It implements four methodological errors that the package pipeline fixes,
> and each one flatters the result:
>
> | defect | in this notebook | why it is wrong |
> |---|---|---|
> | Alphabetical **truncation** of the universe | `sample = tickers[:SAMPLE_N]` | a 500-cap covered roughly A–POOL, not a cross-section of the index |
> | **Point-in-time** filter applied *before* time-series features | `filter_panel_to_pit(labeled, stints)` then features | rolling windows see gaps where a name left the index, so momentum and volatility are computed across holes |
> | Unlabelled rows **dropped** | `long.dropna(subset=["fwd_ret_10d"])` | removes the newest sessions — exactly the rows a live model has to rank |
> | Row-based, **unpurged** `TimeSeriesSplit` | `TimeSeriesSplit(n_splits=...)` | splits cut through a date, and no purge means the label horizon leaks across the boundary |
>
> On identical portfolio rules, the truncated universe showed **+76.6% and Sharpe 1.05** where the corrected panel shows **+22.2% and Sharpe 0.16**. Outputs have been cleared so those figures are not displayed as findings.
>
> For anything current use the package: `train-sp500`, `backtest-sp500`, `predict-sp500`, or the `stock_predictor` API. See the README's Limitations section for what is measured and how.


# S&P 500 weekly move exploration

Goal: explore how rare **≥5% gains over the next 10 trading days** are, and build a clean panel (features at `t`, label from `t+1…t+10`).

Run with: `uv run jupyter notebook` or `uv run jupyter lab` from the project root.

In [ ]:
from __future__ import annotations

import io

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option("display.max_columns", 20)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

from pathlib import Path
import sys

for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "pyproject.toml").exists():
        sys.path.insert(0, str(_root.resolve()))
        break

from sp500_pit import filter_panel_to_pit, load_sp500_stints, tickers_overlapping_window

START = "2018-01-01"
END = None

## 1. S&P 500 universe (point-in-time)

Universe comes from **[fja05680/sp500](https://github.com/fja05680/sp500)** `sp500_ticker_start_end.csv` (community-maintained, not official S&P). Rows are filtered to membership stints `[start_date, end_date)` after prices are stacked.

In [ ]:
SP500_STINTS_URL = (
    "https://raw.githubusercontent.com/fja05680/sp500/master/sp500_ticker_start_end.csv"
)
stints = load_sp500_stints(SP500_STINTS_URL)
tickers = tickers_overlapping_window(stints, START, END)
print(f"Tickers (union of stints overlapping [START, END]): {len(tickers)}")
stints.head()

## 2. Download daily prices (sample first)

Pulling all 500 names on the first run is slow and rate-limit prone. Start with `SAMPLE_N` tickers, then increase.

In [ ]:
SAMPLE_N = 1000000  # raise once you're happy with the pipeline

sample = tickers[:SAMPLE_N]
data = yf.download(
    sample,
    start=START,
    end=END,
    group_by="ticker",
    threads=True,
    auto_adjust=True,
    progress=False,
)

# With auto_adjust=True, yfinance does not return "Adj Close" — OHLC are already adjusted; use "Close".
# With auto_adjust=False, prefer "Adj Close" for total return.
def wide_adj_close(raw: pd.DataFrame) -> pd.DataFrame:
    if isinstance(raw.columns, pd.MultiIndex):
        fields = set(raw.columns.get_level_values(-1))
        price_col = "Adj Close" if "Adj Close" in fields else "Close"
        return raw.xs(price_col, axis=1, level=-1).sort_index()
    if "Adj Close" in raw.columns:
        return raw[["Adj Close"]].sort_index()
    return raw[["Close"]].sort_index()


adj_close = wide_adj_close(data)
adj_close.tail()

## 3. Forward 5-trading-day return and label

For each row (date) per ticker: **close-to-close** return from that close to the close **10 trading sessions later**.
Label `target_5pct` = 1 if forward return ≥ 5%.

In [ ]:
HORIZON = 10
THRESHOLD = 0.05

long = adj_close.stack(future_stack=True).rename("adj_close").reset_index()
long.columns = ["date", "ticker", "adj_close"]
long = long.sort_values(["ticker", "date"])

long["fwd_ret_10d"] = (
    long.groupby("ticker", group_keys=False)["adj_close"].transform(
        lambda s: s.shift(-HORIZON) / s - 1.0
    )
)
long["target_5pct"] = (long["fwd_ret_10d"] >= THRESHOLD).astype("int8")

labeled = long.dropna(subset=["fwd_ret_10d"])
labeled = filter_panel_to_pit(labeled, stints)
labeled.head(10)

## 4. How rare is +5% / 10 sessions?

Expect **class imbalance** (many more 0s than 1s) — that’s what makes ranking and precision@k matter more than raw accuracy.

In [ ]:
pos_rate = labeled["target_5pct"].mean()
print(f"Positive rate (>= {THRESHOLD:.0%} in {HORIZON} sessions): {pos_rate:.4%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
labeled["fwd_ret_10d"].clip(-0.5, 0.5).hist(bins=80, ax=ax[0], color="steelblue", edgecolor="white")
ax[0].set_title("Forward 10d return (clipped ±50%)")
ax[0].set_xlabel("return")
labeled["target_5pct"].value_counts().sort_index().plot(kind="bar", ax=ax[1], color=["#ccc", "#2ca02c"])
ax[1].set_title("Label distribution")
ax[1].set_xticklabels(["0", "1"], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell: Feature Engineering ──────────────────────────────────────────────
# All features use ONLY information available at time `t` to avoid lookahead bias.

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Input:  long-format DataFrame with columns [date, ticker, adj_close]
            sorted by [ticker, date].
    Output: same DataFrame with new feature columns added.
    """
    g = df.groupby("ticker", group_keys=False)["adj_close"]

    # ── Past returns ──────────────────────────────────────────────────────
    for lag in [1, 5, 10, 21]:
        df[f"ret_{lag}d"] = g.transform(lambda s: s.pct_change(lag))

    # ── Momentum (21-day return minus 5-day, avoids short-term reversal) ──
    df["momentum"] = df["ret_21d"] - df["ret_5d"]

    # ── Volatility (rolling std of daily returns) ─────────────────────────
    daily_ret = g.transform(lambda s: s.pct_change())
    df["vol_10d"] = daily_ret.groupby(df["ticker"]).transform(
        lambda s: s.rolling(10).std()
    )
    df["vol_21d"] = daily_ret.groupby(df["ticker"]).transform(
        lambda s: s.rolling(21).std()
    )

    # ── Volume z-score (requires volume data — see note below) ────────────
    # Skipping for now if not available; add when you pull volume from yfinance

    # ── RSI (14-day) ──────────────────────────────────────────────────────
    def rsi(s: pd.Series, window: int = 14) -> pd.Series:
        delta = s.diff()
        gain = delta.clip(lower=0).rolling(window).mean()
        loss = (-delta.clip(upper=0)).rolling(window).mean()
        rs = gain / loss.replace(0, np.nan)
        return 100 - 100 / (1 + rs)

    df["rsi_14"] = g.transform(rsi)

    # ── Price relative to rolling mean (mean-reversion signal) ───────────
    df["price_vs_ma20"] = g.transform(lambda s: s / s.rolling(20).mean() - 1)
    df["price_vs_ma50"] = g.transform(lambda s: s / s.rolling(50).mean() - 1)

    # ── Bollinger Band position ───────────────────────────────────────────
    def bb_position(s: pd.Series, window: int = 20) -> pd.Series:
        ma = s.rolling(window).mean()
        std = s.rolling(window).std()
        upper = ma + 2 * std
        lower = ma - 2 * std
        return (s - lower) / (upper - lower + 1e-9)

    df["bb_pos"] = g.transform(bb_position)

    return df


features = add_features(labeled.copy())

FEATURE_COLS = [
    "ret_1d", "ret_5d", "ret_10d", "ret_21d",
    "momentum", "vol_10d", "vol_21d",
    "rsi_14", "price_vs_ma20", "price_vs_ma50", "bb_pos",
]

# Drop rows with NaN in any feature (warm-up period for rolling windows)
features = features.dropna(subset=FEATURE_COLS + ["target_5pct"])
print(f"Panel shape: {features.shape}")
print(f"Positive rate after feature engineering: {features['target_5pct'].mean():.4%}")
features[FEATURE_COLS].describe()

In [ ]:
# ── Cell: Time-based split ──────────────────────────────────────────────────
# CRITICAL: never use random splits for time series — it leaks future information.

TRAIN_END = "2022-12-31"
TEST_START = "2023-01-01"

train = features[features["date"] <= TRAIN_END]
test  = features[features["date"] >= TEST_START]

X_train, y_train = train[FEATURE_COLS], train["target_5pct"]
X_test,  y_test  = test[FEATURE_COLS],  test["target_5pct"]

print(f"Train: {X_train.shape} | Positives: {y_train.mean():.4%}")
print(f"Test:  {X_test.shape}  | Positives: {y_test.mean():.4%}")

In [ ]:
# ── Cell: LightGBM model ────────────────────────────────────────────────────
# uv add lightgbm scikit-learn  (if not already installed)

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    roc_auc_score,
)

# scale_pos_weight handles class imbalance: ratio of negatives to positives
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight: {spw:.1f}")

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    scale_pos_weight=spw,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

In [ ]:
# ── Cell: Evaluation ────────────────────────────────────────────────────────
# With heavy class imbalance, accuracy is meaningless.
# We care about:
#   1. PR-AUC: overall ranking quality for the positive class
#   2. Precision@k: when we pick the top-k predictions per week, how often are we right?

y_prob = model.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)
print(f"PR-AUC  : {pr_auc:.4f}  (baseline = {y_test.mean():.4f})")
print(f"ROC-AUC : {roc_auc:.4f}  (baseline = 0.5000)")

# ── Precision@k (the realistic trading metric) ────────────────────────────
def precision_at_k(y_true, y_scores, k: int) -> float:
    """Of the top-k ranked predictions, what fraction are truly positive?"""
    top_k_idx = np.argsort(y_scores)[-k:]
    return y_true.iloc[top_k_idx].mean()

test_scored = test.copy()
test_scored["prob"] = y_prob

# Simulate: every week, pick the top-10 candidates and check precision
weekly_precision = (
    test_scored
    .assign(week=lambda df: pd.to_datetime(df["date"]).dt.to_period("W"))
    .groupby("week")
    .apply(lambda g: precision_at_k(g["target_5pct"], g["prob"], k=10))
)

print(f"\nMean weekly Precision@10 : {weekly_precision.mean():.4f}")
print(f"Baseline (random pick)   : {y_test.mean():.4f}")

weekly_precision.plot(figsize=(12, 3), title="Weekly Precision@10 on Test Set")
plt.axhline(y_test.mean(), color="red", linestyle="--", label="baseline")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell: Feature importance ─────────────────────────────────────────────────
importance = pd.Series(
    model.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=True)

importance.plot(kind="barh", figsize=(8, 5), title="LightGBM Feature Importance (gain)")
plt.tight_layout()
plt.show()

## Next steps (when you’re ready)

- Add **features** known at `date` only (past returns, vol, volume z-scores).
- Use **time-based splits**; measure **precision@k** and PR-AUC, not plain accuracy.
- Swap the stint CSV URL for a **vendor feed** (e.g. WRDS/Compustat) if you need official S&P history.